In [6]:
import os
import urllib.request
import time
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Directory & Hardware Setup
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using execution device: {device}\n")


# 2. Download & Load Indian Pines Dataset
data_url = "https://raw.githubusercontent.com/ZhangQuan-hub/HyperspectralDatasets/main/Indian_pines_corrected.mat"
gt_url = "https://raw.githubusercontent.com/fhung65/math156FinalProject/master/Indian_pines_gt.mat"

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

if not os.path.exists("Indian_pines_corrected.mat"):
    print("Downloading Indian Pines spectral cube...")
    req = urllib.request.Request(data_url, headers=headers)
    with urllib.request.urlopen(req) as response, open("Indian_pines_corrected.mat", "wb") as out_file:
        out_file.write(response.read())

if not os.path.exists("Indian_pines_gt.mat"):
    print("Downloading ground truth labels...")
    req = urllib.request.Request(gt_url, headers=headers)
    with urllib.request.urlopen(req) as response, open("Indian_pines_gt.mat", "wb") as out_file:
        out_file.write(response.read())

data = sio.loadmat("Indian_pines_corrected.mat")['indian_pines_corrected']
gt = sio.loadmat("Indian_pines_gt.mat")['indian_pines_gt']

# Min-Max Normalization across spectral bands
data = data.astype(np.float32)
data = (data - data.min()) / (data.max() - data.min())
# Min-Max Normalization across spectral bands
data = data.astype(np.float32)
data = (data - data.min()) / (data.max() - data.min())

# Min-Max Normalization across spectral bands
data = data.astype(np.float32)
data = (data - data.min()) / (data.max() - data.min())

# 3. Extract 3D Spatial-Spectral Patches
patch_size = 9
pad = patch_size // 2
padded_data = np.pad(data, ((pad, pad), (pad, pad), (0, 0)), mode='constant')

X, Y = [], []
H, W, B = data.shape

for r in range(H):
    for c in range(W):
        label = gt[r, c]
        if label > 0:  # Ignore background (0)
            patch = padded_data[r:r+patch_size, c:c+patch_size, :]
            X.append(patch)
            Y.append(label - 1)  # Shift labels 1-16 to 0-15

X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.int64)

# Reshape for 3D Conv: (N, Channels=1, Depth=Bands, Height=9, Width=9)
X = np.transpose(X, (0, 3, 1, 2))
X = np.expand_dims(X, axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y)

class HSIDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x = torch.tensor(x_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = HSIDataset(X_train, y_train)
test_ds = HSIDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

# 4. Define 3D Convolutional Neural Network
class Hyperspectral3DCNN(nn.Module):
    def __init__(self, num_classes=16):
        super(Hyperspectral3DCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=(7, 3, 3), padding=(0, 1, 1)),
            nn.BatchNorm3d(8),
            nn.ReLU(),
            nn.MaxPool3d((2, 1, 1)),

            nn.Conv3d(8, 16, kernel_size=(5, 3, 3), padding=(0, 1, 1)),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d((2, 1, 1)),

            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=(0, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((4, 2, 2))
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 4 * 2 * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# 5. Training & Evaluation
epochs = 35
batch_size = 32

# Re-create DataLoaders with smaller batch size for better convergence
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

model = Hyperspectral3DCNN(num_classes=16).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Added scheduler to smooth out validation spikes
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

train_losses, val_losses, val_accuracies = [], [], []

print(f"Successfully prepared {len(X)} spatial-spectral samples ({B} bands).")
print(f"Starting optimized 3D-CNN training for {epochs} epochs...\n")

best_val_acc = 0.0

for epoch in range(epochs):
    start_time = time.time()

    # Train Phase
    model.train()
    running_train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_train_loss / len(train_ds)
    train_losses.append(epoch_train_loss)

    # Validation Phase
    model.eval()
    running_val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = running_val_loss / len(test_ds)
    val_acc = correct / total

    val_losses.append(epoch_val_loss)
    val_accuracies.append(val_acc)

    # Save the absolute best weights
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "models/hyperspectral_3d_cnn.pth")

    scheduler.step()
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | {elapsed:4.1f}s | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f})")

print(f"\nTraining completed! Highest Validation Accuracy: {best_val_acc * 100:.2f}%")
# 6. Plot 1: Training Metrics
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), train_losses, label='Train Loss')
plt.plot(range(1, epochs + 1), val_losses, label='Val Loss')
plt.title('3D-CNN Loss Progression')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), val_accuracies, color='g', label='Val Accuracy')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy Ratio')
plt.grid(True)

plt.tight_layout()
plt.savefig("results/training_metrics.png", dpi=300)
plt.close()
print("Saved training metrics plot to: results/training_metrics.png")

# 7. Plot 2: Full Spatial Classification Map
model.eval()
pred_map = np.zeros((H, W))

with torch.no_grad():
    for r in range(H):
        for c in range(W):
            if gt[r, c] > 0:
                patch = padded_data[r:r+patch_size, c:c+patch_size, :]
                patch = np.transpose(patch, (2, 0, 1))
                patch = np.expand_dims(patch, axis=(0, 1))
                patch_tensor = torch.tensor(patch, dtype=torch.float32).to(device)

                output = model(patch_tensor)
                _, pred = torch.max(output, 1)
                pred_map[r, c] = pred.item() + 1

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(gt, cmap='nipy_spectral')
axes[0].set_title("Ground Truth Map")
axes[0].axis('off')

axes[1].imshow(pred_map, cmap='nipy_spectral')
axes[1].set_title("3D-CNN Predicted Map")
axes[1].axis('off')

plt.tight_layout()
plt.savefig("results/classification_results.png", dpi=300)
plt.close()
print("Saved full spatial classification map to: results/classification_results.png")

Using execution device: cuda:0

Successfully prepared 10249 spatial-spectral samples (200 bands).
Starting optimized 3D-CNN training for 35 epochs...

Epoch 01/35 |  7.3s | Train Loss: 1.7469 | Val Loss: 1.4138 | Val Acc: 0.4696 (Best: 0.4696)
Epoch 02/35 |  7.2s | Train Loss: 1.3684 | Val Loss: 1.3944 | Val Acc: 0.4859 (Best: 0.4859)
Epoch 03/35 |  7.2s | Train Loss: 1.2377 | Val Loss: 1.1611 | Val Acc: 0.5610 (Best: 0.5610)
Epoch 04/35 |  7.3s | Train Loss: 1.1589 | Val Loss: 1.4429 | Val Acc: 0.4933 (Best: 0.5610)
Epoch 05/35 |  7.3s | Train Loss: 1.0640 | Val Loss: 1.0346 | Val Acc: 0.5987 (Best: 0.5987)
Epoch 06/35 |  7.2s | Train Loss: 1.0043 | Val Loss: 1.0700 | Val Acc: 0.5733 (Best: 0.5987)
Epoch 07/35 |  7.2s | Train Loss: 0.9367 | Val Loss: 0.8132 | Val Acc: 0.6914 (Best: 0.6914)
Epoch 08/35 |  7.1s | Train Loss: 0.8737 | Val Loss: 0.7683 | Val Acc: 0.6995 (Best: 0.6995)
Epoch 09/35 |  7.2s | Train Loss: 0.8166 | Val Loss: 0.8310 | Val Acc: 0.7089 (Best: 0.7089)
Epoch 10/35 